# Tabular Deep Learning - FX Pairs

TabM applies a small neural network to each decision row without turning the history into a
sequence. This notebook submits the published capacity choices to the shared TabM runner. The
runner fits preprocessing inside each training fold, saves every declared weight checkpoint, and
publishes a separate complete validation prediction set for every checkpoint.

**Learning objectives**

- Express neural-network capacity and checkpoint schedules as visible requests.
- Verify that every fold and epoch checkpoint has reloadable fitted state.
- Continue from complete prediction rows without selecting a checkpoint by rank correlation.

**Book reference**: Chapter 12, Section 12.3

**Prerequisites**: `02_labels`, `03_financial_features`, and `04_model_based_features`.

In [1]:
"""Fit and catalog the published TabM FX configurations."""

import polars as pl
import torch
import yaml

from case_studies.research import ExecutionTier, Study, plan_models
from utils.modeling import load_configs
from utils.paths import get_case_study_dir
from utils.reproducibility import set_global_seeds

In [2]:
CASE_STUDY_ID = "fx_pairs"
PRIMARY_LABEL = ""
MAX_SYMBOLS = 0
MAX_FOLDS = 0
FORCE_RETRAIN = False
PREDICTION_SPLIT = "validation"
N_EPOCHS = 0
BATCH_SIZE = 0
DEVICE = ""
SEED = 42

## Select the task and execution tier

Canonical execution uses every configured fold, symbol, epoch, and batch setting. Supplying a
reduction creates a preview identity in an isolated registry. A preview proves the path but cannot
join the official model population.

In [3]:
set_global_seeds(SEED)
case_dir = get_case_study_dir(CASE_STUDY_ID)
setup = yaml.safe_load((case_dir / "config" / "setup.yaml").read_text())
labels = (
    [PRIMARY_LABEL]
    if PRIMARY_LABEL
    else [setup["labels"]["primary"], *setup["labels"].get("variants", [])]
)

if PREDICTION_SPLIT != "validation":
    raise ValueError("model selection uses validation predictions; holdout runs start from a lock")
if FORCE_RETRAIN:
    raise ValueError("valid checkpoints are reloaded by identity; change the request to refit")

reductions = {
    **({"folds": list(range(MAX_FOLDS))} if MAX_FOLDS else {}),
    **({"max_symbols": MAX_SYMBOLS} if MAX_SYMBOLS else {}),
    **({"n_epochs": N_EPOCHS} if N_EPOCHS else {}),
}
tier = ExecutionTier.PREVIEW if reductions else ExecutionTier.CANONICAL
study = Study.regenerate(CASE_STUDY_ID)

print(f"Labels: {', '.join(labels)}")
print(f"Execution tier: {tier.value}")
# An empty DEVICE resolves to what the machine has. The runners refuse "cuda" on a host without
# it rather than falling back silently - which is the right contract for a run whose results get
# registered - so a hardcoded "cuda" default made the notebook unrunnable for any reader without
# an NVIDIA card, and unrunnable on a CPU CI runner. Resolving here keeps the refusal for anyone
# who asks for "cuda" explicitly, and prints what was chosen so a run never leaves it implicit.
device = DEVICE or ("cuda" if torch.cuda.is_available() else "cpu")
print(f"Device: {device}")

Labels: fwd_ret_1d, fwd_ret_5d, fwd_ret_21d
Execution tier: canonical
Device: cuda


## Build the published requests

The YAML menu supplies the architecture settings and production checkpoint schedules. The
parameter cell can reduce epochs or change batch size for a preview without changing the menu.

In [4]:
overrides = {
    "device": device,
    **({"batch_size": BATCH_SIZE} if BATCH_SIZE else {}),
}
menu = [
    (label, config)
    for label in labels
    for config in load_configs(CASE_STUDY_ID, label, family="tabular_dl")
]
requests = [
    study.model(
        family="tabular_dl",
        label=label,
        config_name=config["config_name"],
        execution_tier=tier,
        preview_reductions=reductions,
        overrides=overrides,
    )
    for label, config in menu
]

pl.DataFrame(
    {
        "config_name": [request.config_name for request in requests],
        "label": [request.label for request in requests],
        "device": [device] * len(requests),
        "execution_tier": [request.execution_tier.value for request in requests],
    }
)

config_name,label,device,execution_tier
str,str,str,str
"""tabm_s""","""fwd_ret_1d""","""cuda""","""canonical"""
"""tabm_m""","""fwd_ret_1d""","""cuda""","""canonical"""
"""tabm_l""","""fwd_ret_1d""","""cuda""","""canonical"""
"""tabm_s""","""fwd_ret_5d""","""cuda""","""canonical"""
"""tabm_m""","""fwd_ret_5d""","""cuda""","""canonical"""
"""tabm_l""","""fwd_ret_5d""","""cuda""","""canonical"""
"""tabm_s""","""fwd_ret_21d""","""cuda""","""canonical"""
"""tabm_m""","""fwd_ret_21d""","""cuda""","""canonical"""
"""tabm_l""","""fwd_ret_21d""","""cuda""","""canonical"""


## Declare every epoch checkpoint before training

The declared epoch schedule, not the run that follows, decides how many downstream configurations
this notebook owes. Planning resolves each one without training, so a failed member is visible as
a gap in the population rather than a shorter catalog.

In [5]:
plan = plan_models(study, requests=requests)
if len(plan.expected_training_hashes) != len(requests):
    raise RuntimeError("each TabM configuration must plan exactly one training identity")

configured = {(label, config["config_name"]) for label, config in menu}
planned = {(member.label, member.config_name) for member in plan.members}
if planned != configured:
    raise RuntimeError(
        "the plan does not match the configured TabM menu; "
        f"missing {sorted(configured - planned)}, unexpected {sorted(planned - configured)}"
    )

pl.DataFrame(
    {
        "label": [member.label for member in plan.members],
        "config_name": [member.config_name for member in plan.members],
        "checkpoint_kind": [member.checkpoint_kind for member in plan.members],
        "checkpoint_value": [member.checkpoint_value for member in plan.members],
        "prediction_hash": [member.prediction_hash for member in plan.members],
    }
)

label,config_name,checkpoint_kind,checkpoint_value,prediction_hash
str,str,str,i64,str
"""fwd_ret_1d""","""tabm_s""","""epoch""",25,"""243c981790e5"""
"""fwd_ret_1d""","""tabm_s""","""epoch""",50,"""fce2b00758da"""
"""fwd_ret_1d""","""tabm_s""","""epoch""",75,"""91868fa741e6"""
"""fwd_ret_1d""","""tabm_s""","""epoch""",100,"""7bfe5132ffdf"""
"""fwd_ret_1d""","""tabm_s""","""epoch""",125,"""18cee173f0dc"""
…,…,…,…,…
"""fwd_ret_21d""","""tabm_l""","""epoch""",100,"""dc71fa146a3d"""
"""fwd_ret_21d""","""tabm_l""","""epoch""",125,"""03e9540dbeb8"""
"""fwd_ret_21d""","""tabm_l""","""epoch""",150,"""615228463a26"""


## Record the official population, then fit or reload every capacity choice

Compatible TabM requests share base-fold materialization. Candidate-specific scaling, random
state, weights, and prediction identities remain separate. Any failed member stops the cell.

In [6]:
population = (
    plan.create_population(name=f"{CASE_STUDY_ID}:{'+'.join(labels)}:tabular_dl")
    if tier is ExecutionTier.CANONICAL
    else None
)

execution = plan.run()
if len(execution.runs) != len(requests):
    raise RuntimeError("the TabM runner did not return every requested configuration")

catalog = execution.catalog_rows.sort("label", "config_name", "checkpoint_value")
if set(catalog.get_column("prediction_hash")) != set(plan.expected_prediction_hashes):
    raise RuntimeError("the published catalog differs from the population planned before fitting")
if catalog.filter(~pl.col("complete")).height:
    raise RuntimeError("partial TabM checkpoints cannot pass to backtesting")
if catalog.select("label", "config_name", "checkpoint_value").n_unique() != catalog.height:
    raise RuntimeError("each configuration and epoch checkpoint must identify one prediction set")
if catalog.get_column("checkpoint_value").null_count():
    raise RuntimeError("every TabM prediction must name its epoch checkpoint")

catalog.select(
    "label",
    "config_name",
    "checkpoint_kind",
    "checkpoint_value",
    "complete",
    "ic_mean",
    "ic_t",
    "training_hash",
    "prediction_hash",
)

label,config_name,checkpoint_kind,checkpoint_value,complete,ic_mean,ic_t,training_hash,prediction_hash
str,str,str,i64,bool,f64,f64,str,str
"""fwd_ret_1d""","""tabm_l""","""epoch""",25,true,-0.004033,-0.526438,"""2b929abc9475""","""f9155c11faba"""
"""fwd_ret_1d""","""tabm_l""","""epoch""",50,true,0.006267,0.924793,"""2b929abc9475""","""d937c16f7118"""
"""fwd_ret_1d""","""tabm_l""","""epoch""",75,true,0.002941,0.531829,"""2b929abc9475""","""e1e57ac87164"""
"""fwd_ret_1d""","""tabm_l""","""epoch""",100,true,0.004194,0.584408,"""2b929abc9475""","""83b706ea3f17"""
"""fwd_ret_1d""","""tabm_l""","""epoch""",125,true,0.004526,0.607901,"""2b929abc9475""","""9deaf63b1d24"""
…,…,…,…,…,…,…,…,…
"""fwd_ret_5d""","""tabm_s""","""epoch""",100,true,-0.011937,-1.15802,"""f960af626672""","""7d0f251f9e43"""
"""fwd_ret_5d""","""tabm_s""","""epoch""",125,true,-0.01185,-0.981717,"""f960af626672""","""190a7323d12d"""
"""fwd_ret_5d""","""tabm_s""","""epoch""",150,true,-0.012756,-1.051666,"""f960af626672""","""a7dd3a9eeabd"""


## Reload the checkpoint population

Repeating the same request validates the saved checkpoint manifests and returns the same catalog
identities. No empty cached summary or single IC-chosen checkpoint is substituted.

In [7]:
replayed = plan.run()
if set(replayed.catalog_rows.get_column("prediction_hash")) != set(
    catalog.get_column("prediction_hash")
):
    raise RuntimeError("TabM checkpoint reload changed the prediction population")

if population is not None:
    population.require_complete()
    print(f"Official prediction population: {population.hash}")
else:
    print("Preview checkpoints remain outside official comparison and holdout selection.")

Official prediction population: 0044e1f31b9c


## Key takeaways

- Train-only preprocessing and checkpoint persistence belong to the shared TabM computation.
- Every declared epoch remains available to the backtest stage.
- Rank correlation is a diagnostic field in the catalog, not a checkpoint-selection rule.